0️⃣ Imports

In [1]:
import numpy as np
from numpy.linalg import inv, solve
from scipy.linalg import cho_factor, cho_solve


1️⃣ Simulation (ground truth)

In [2]:
def simulate_ou_dfa(
    N=200,
    T=4,
    p=5000,
    K=20,
    rho=0.5,
    sigma_q=0.3,
    sigma_y=1.0,
    seed=0,
):
    rng = np.random.default_rng(seed)

    # Sparse loadings
    Lambda = rng.normal(0, 0.2, size=(p, K))
    mask = rng.random((p, K)) < 0.9
    Lambda[mask] = 0.0

    Q = sigma_q**2 * np.eye(K)
    Psi = sigma_y**2 * np.ones(p)

    Y_all, times_all, Z_all = [], [], []

    for i in range(N):
        times = np.sort(rng.uniform(30, 80, size=T))
        dts = np.diff(times)

        z = np.zeros((T, K))
        z[0] = rng.normal(0, 1, size=K)

        for t in range(1, T):
            A = np.exp(-rho * dts[t-1])
            z[t] = A * z[t-1] + rng.multivariate_normal(np.zeros(K), Q)

        y = z @ Lambda.T + rng.normal(0, sigma_y, size=(T, p))

        Y_all.append(y)
        times_all.append(times)
        Z_all.append(z)

    return Y_all, times_all, Lambda, Z_all


2️⃣ OU transition

In [3]:
def ou_A(rho, dt):
    return np.exp(-rho * dt)


3️⃣ Kalman smoother (E-step)

In [4]:
def kalman_smoother(Y, times, Lambda, Q, Psi, rho):
    T, p = Y.shape
    K = Lambda.shape[1]

    R = np.diag(Psi)
    H = Lambda

    m = np.zeros((T, K))
    P = np.zeros((T, K, K))
    m_pred = np.zeros_like(m)
    P_pred = np.zeros_like(P)

    m[0] = 0
    P[0] = np.eye(K)

    # Forward
    for t in range(1, T):
        A = ou_A(rho, times[t] - times[t-1]) * np.eye(K)
        m_pred[t] = A @ m[t-1]
        P_pred[t] = A @ P[t-1] @ A.T + Q

        S = H @ P_pred[t] @ H.T + R
        c, low = cho_factor(S, lower=True)
        Kt = cho_solve((c, low), H @ P_pred[t].T).T

        m[t] = m_pred[t] + Kt @ (Y[t] - H @ m_pred[t])
        P[t] = (np.eye(K) - Kt @ H) @ P_pred[t]

    # RTS smoother
    Ez = m.copy()
    Ezz = np.zeros_like(P)
    Ezz_lag = np.zeros_like(P)

    for t in reversed(range(T-1)):
        A = ou_A(rho, times[t+1] - times[t]) * np.eye(K)
        C = P[t] @ A.T @ inv(P_pred[t+1])

        Ez[t] += C @ (Ez[t+1] - m_pred[t+1])
        P[t] += C @ (P[t+1] - P_pred[t+1]) @ C.T
        Ezz_lag[t+1] = P[t+1] @ C.T + np.outer(Ez[t+1], Ez[t])

    for t in range(T):
        Ezz[t] = P[t] + np.outer(Ez[t], Ez[t])

    return Ez, Ezz, Ezz_lag


4️⃣ Horseshoe MAP update for $\Lambda$ (ECM)

In [5]:
def update_lambda_horseshoe(Ez_all, Y_all, tau, lam):
    Z = np.vstack(Ez_all)
    Y = np.vstack(Y_all)

    p = Y.shape[1]
    K = Z.shape[1]

    Lambda = np.zeros((p, K))

    for j in range(p):
        Dinv = np.diag(1.0 / (tau * lam[j])**2)
        Sj = Z.T @ Z + Dinv
        bj = Z.T @ Y[:, j]
        Lambda[j] = solve(Sj, bj)

    return Lambda


5️⃣ Noise update

In [6]:
def update_noise(Y_all, Ez_all, Lambda):
    resid = []
    for Y, Ez in zip(Y_all, Ez_all):
        resid.append(Y - Ez @ Lambda.T)
    resid = np.vstack(resid)
    return np.var(resid, axis=0) + 1e-6


6️⃣ OU parameter update

In [7]:
def update_rho(Ezz_all, Ezz_lag_all, times_all):
    num, den = 0.0, 0.0
    for Ezz, Ezz_lag, times in zip(Ezz_all, Ezz_lag_all, times_all):
        for t in range(1, len(times)):
            num += np.trace(Ezz_lag[t])
            den += np.trace(Ezz[t-1])
    rho = -np.log(num / den) / np.mean([np.diff(t).mean() for t in times_all])
    return max(rho, 1e-3)


3️⃣ FIX 1: Correct OU parameter updates

In [ ]:
def update_Q(Ezz, Ezz_lag, rho, dts):
    num = 0.0
    den = 0
    for i in range(len(Ezz)):
        for t in range(1, len(Ezz[i])):
            a = np.exp(-rho * dts[i][t-1])
            num += Ezz[i][t] - a * Ezz_lag[i][t-1]
            den += 1
    return num / den


4️⃣ FIX 2: Proper horseshoe MAP (global–local shrinkage)

In [ ]:
def update_local_scales(Lambda, eps=1e-6):
    return 1.0 / (Lambda**2 + eps)


In [ ]:
def update_global_scale(Lambda):
    return np.median(np.abs(Lambda))


In [ ]:
def update_Lambda(Ez, Ezz, Y, Psi, tau, lam):
    K = Ez.shape[1]
    reg = np.diag(tau**2 * lam)
    XtX = Ezz + reg
    XtY = Ez.T @ Y
    return np.linalg.solve(XtX, XtY).T


7️⃣ Full EM loop

In [8]:
def fit_ou_dfa(Y_all, times_all, K, max_iter=30):
    p = Y_all[0].shape[1]

    Lambda = np.random.randn(p, K) * 0.01
    Psi = np.ones(p)
    Q = np.eye(K) * 0.1
    rho = 0.3

    tau = 1.0
    lam = np.ones((p, K))

    for it in range(max_iter):
        Ez_all, Ezz_all, Ezz_lag_all = [], [], []

        # E-step
        for Y, times in zip(Y_all, times_all):
            Ez, Ezz, Ezz_lag = kalman_smoother(Y, times, Lambda, Q, Psi, rho)
            Ez_all.append(Ez)
            Ezz_all.append(Ezz)
            Ezz_lag_all.append(Ezz_lag)

        # M-step
        Lambda = update_lambda_horseshoe(Ez_all, Y_all, tau, lam)
        Psi = update_noise(Y_all, Ez_all, Lambda)
        rho = update_rho(Ezz_all, Ezz_lag_all, times_all)

        print(
            f"Iter {it:02d} | "
            f"rho={rho:.3f} | "
            f"mean|Λ|={np.mean(np.abs(Lambda)):.4f}"
        )

    return Lambda, Psi, rho


8️⃣ Validation utilities

In [ ]:
from scipy.linalg import orthogonal_procrustes

def factor_recovery(L_true, L_est):
    R, _ = orthogonal_procrustes(L_est, L_true)
    L_aligned = L_est @ R
    C = np.abs(np.corrcoef(
        L_true.T, L_aligned.T
    )[:L_true.shape[1], L_true.shape[1]:])
    return np.mean(np.max(C, axis=1))


In [11]:
from scipy.linalg import orthogonal_procrustes

def aligned_recovery(L_true, L_est):
    R, _ = orthogonal_procrustes(L_est, L_true)
    L_aligned = L_est @ R
    corr = np.abs(np.corrcoef(
        L_true.T, L_aligned.T
    )[:L_true.shape[1], L_true.shape[1]:])
    return np.mean(np.max(corr, axis=1))


9️⃣ Main (end-to-end run)

In [12]:
def main():
    Y_all, times_all, Lambda_true, Z_true = simulate_ou_dfa(
        N=100,
        T=4,
        p=2000,   # increase to 10_000 once tested
        K=20,
        seed=0
    )

    Lambda_est, Psi_est, rho_est = fit_ou_dfa(
        Y_all,
        times_all,
        K=20,
        max_iter=20
    )

    # rec = factor_recovery(Lambda_true, Lambda_est)
    rec = aligned_recovery(Lambda_true, Lambda_est)
    print("Factor recovery (mean corr):", rec.mean())


if __name__ == "__main__":
    main()


Iter 00 | rho=0.159 | mean|Λ|=0.3532
Iter 01 | rho=0.277 | mean|Λ|=0.3063
Iter 02 | rho=0.289 | mean|Λ|=0.2790
Iter 03 | rho=0.280 | mean|Λ|=0.2568
Iter 04 | rho=0.267 | mean|Λ|=0.2387
Iter 05 | rho=0.253 | mean|Λ|=0.2238
Iter 06 | rho=0.239 | mean|Λ|=0.2114
Iter 07 | rho=0.226 | mean|Λ|=0.2010
Iter 08 | rho=0.214 | mean|Λ|=0.1922
Iter 09 | rho=0.203 | mean|Λ|=0.1848
Iter 10 | rho=0.193 | mean|Λ|=0.1786
Iter 11 | rho=0.185 | mean|Λ|=0.1734
Iter 12 | rho=0.177 | mean|Λ|=0.1689
Iter 13 | rho=0.171 | mean|Λ|=0.1651
Iter 14 | rho=0.165 | mean|Λ|=0.1618
Iter 15 | rho=0.160 | mean|Λ|=0.1591
Iter 16 | rho=0.155 | mean|Λ|=0.1566
Iter 17 | rho=0.150 | mean|Λ|=0.1546
Iter 18 | rho=0.146 | mean|Λ|=0.1527
Iter 19 | rho=0.143 | mean|Λ|=0.1511
Factor recovery (mean corr): 0.24352765752968994
